In [1]:
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

In [2]:
train_df = pd.read_parquet("data/df_train_preprocessed.parquet")
test_df = pd.read_parquet("data/df_test_preprocessed.parquet")

id_cols = ["month_decision", "weekday_decision", "WEEK_NUM", "case_id"]

# Guarantee chronological row order before dropping WEEK_NUM bcs TimeSeriesSplit has this as the assumption
train_df = train_df.sort_values(
    "WEEK_NUM", kind="mergesort").reset_index(drop=True)
test_df = test_df.sort_values(
    "WEEK_NUM", kind="mergesort").reset_index(drop=True)

# Keep weeknum for our stratisfied sampling during tuning
week_num_train = train_df["WEEK_NUM"].copy()

train_df.drop(columns=id_cols, inplace=True)
test_df.drop(columns=id_cols, inplace=True)

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Train shape: (1088836, 202), Test shape: (437823, 202)


## 3.0 Setup

In [3]:
# Define features and target variable
X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

In [ ]:
# ── Shared tuning setup (used by all three models) ──────────────────────────
# NOTE: TimeSeriesSplit relies on row order. X_train / y_train are assumed to be
# already sorted chronologically (by decision time / WEEK_NUM) upstream, so the
# folds below respect the temporal ordering of the training block.

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

N_SPLITS = 5      # TimeSeriesSplit folds used during tuning
# Optuna trials per model (raise/lower for your compute budget)
N_TRIALS = 20

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# ── Fixed class weight ──────────────────────────────────────────────────────
# Computed ONCE on the FULL training block and shared, unchanged, across all
# three models. It is NOT recomputed per fold or per trial.
#
# Documented design choice (not a bug): because weight_ratio comes from the full
# training block, the earliest TimeSeriesSplit folds during tuning are scored
# against a class ratio that is partly informed by *later* training data. We
# accept this deliberately — a single fixed, shared weight keeps the three
# models directly comparable and matches the frozen-model evaluation design.
n_positive = int((y_train == 1).sum())
n_negative = int((y_train == 0).sum())
weight_ratio = n_negative / n_positive
print(f"weight_ratio (n_negative / n_positive) = {weight_ratio:.4f}")


def gini(y_true, y_score):
    """Gini coefficient = 2 * AUC - 1"""
    return 2.0 * roc_auc_score(y_true, y_score) - 1.0


def print_trial(study, trial):
    """Optuna callback: print the Gini and params of each completed trial."""
    print(
        f"  Trial {trial.number:>3} | Gini: {trial.value:.4f} | Params: {trial.params}")


def make_tuning_subsample(X, y, week_num, frac, seed=SEED):
    """Optional stratified subsample of the training block, for TUNING ONLY.

    Stratifies jointly on (WEEK_NUM, target): groups rows by week and class,
    draws frac of the rows within each group, then restores the original row
    order (by position). Returns the data unchanged when frac is None or >= 1.
    """
    if frac is None or frac >= 1.0:
        return X, y
    rng = np.random.default_rng(seed)
    y_arr = y.to_numpy()
    week_arr = week_num.to_numpy()

    keep_parts = []
    for week in np.unique(week_arr):
        week_mask = week_arr == week
        for cls in np.unique(y_arr):
            idx = np.where(week_mask & (y_arr == cls))[0]
            n_keep = max(1, round(len(idx) * frac)) if len(idx) > 0 else 0
            if n_keep > 0:
                keep_parts.append(rng.choice(idx, size=n_keep, replace=False))
    keep = np.sort(np.concatenate(keep_parts))
    return X.iloc[keep], y.iloc[keep]


# Fraction of the training block used during tuning to reduce compute cost. The final model is trained on the full training block.
FRAC_LR = 0.2
FRAC_XGB = 0.2
FRAC_MLP = 0.2

weight_ratio (n_negative / n_positive) = 31.3885


## 3.1 Logistische regressie

In [ ]:
# Optuna tuning — Logistic Regression
X_lr, y_lr = make_tuning_subsample(X_train, y_train, week_num_train, FRAC_LR)


def lr_objective(trial):
    C = trial.suggest_float("C", 1e-4, 1e2, log=True)
    l1_ratio = trial.suggest_categorical("l1_ratio", [0.0, 1.0])
    solver = "newton-cholesky" if l1_ratio == 0.0 else "saga"

    fold_ginis = []
    for tr_idx, va_idx in tscv.split(X_lr):
        model = LogisticRegression(
            C=C,
            l1_ratio=l1_ratio,
            solver=solver,
            class_weight={0: 1.0, 1: weight_ratio},
            max_iter=1000,
            random_state=SEED,
            tol=1e-3,
            warm_start=True
        )
        model.fit(X_lr.iloc[tr_idx], y_lr.iloc[tr_idx])
        proba = model.predict_proba(X_lr.iloc[va_idx])[:, 1]
        fold_ginis.append(gini(y_lr.iloc[va_idx], proba))
    return float(np.mean(fold_ginis))


lr_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
lr_study.optimize(lr_objective, n_trials=N_TRIALS, callbacks=[print_trial], n_jobs=-1)

print("Best LR params:", lr_study.best_params)
print(f"Best LR mean CV Gini: {lr_study.best_value:.4f}")

# Retrain the final LR on the FULL training block with the selected params.
lr_best = LogisticRegression(
    **lr_study.best_params,
    solver="saga", #TODO: check if this is correct, saga is only for l1_ratio > 0
    class_weight={0: 1.0, 1: weight_ratio},
    max_iter=1000,
    random_state=SEED,
    tol=1e-3,
)
lr_best.fit(X_train, y_train)

## 3.2 XGBoost

In [5]:
# Optuna tuning — XGBoost
X_xgb, y_xgb = make_tuning_subsample(
    X_train, y_train, week_num_train, FRAC_XGB)


def xgb_objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    fold_ginis = []
    for tr_idx, va_idx in tscv.split(X_xgb):
        model = XGBClassifier(
            **params,
            scale_pos_weight=weight_ratio,  # fixed, shared weight (not tuned)
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
        )
        model.fit(X_xgb.iloc[tr_idx], y_xgb.iloc[tr_idx])
        proba = model.predict_proba(X_xgb.iloc[va_idx])[:, 1]
        fold_ginis.append(gini(y_xgb.iloc[va_idx], proba))
    return float(np.mean(fold_ginis))


xgb_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, callbacks=[print_trial])

print("Best XGB params:", xgb_study.best_params)
print(f"Best XGB mean CV Gini: {xgb_study.best_value:.4f}")

# Retrain the final XGBoost on the FULL training block with the selected params.
xgb_best = XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=weight_ratio,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)
xgb_best.fit(X_train, y_train)

  Trial   0 | Gini: 0.4643 | Params: {'max_depth': 5, 'learning_rate': 0.22648248189516848, 'n_estimators': 750, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 4, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}
  Trial   1 | Gini: 0.5749 | Params: {'max_depth': 7, 'learning_rate': 0.05675206026988748, 'n_estimators': 100, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'min_child_weight': 5, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}
  Trial   2 | Gini: 0.6085 | Params: {'max_depth': 5, 'learning_rate': 0.0199473547030745, 'n_estimators': 500, 'subsample': 0.645614570099021, 'colsample_bytree': 0.8059264473611898, 'min_child_weight': 3, 'reg_alpha': 4.258943089524393e-06, 'reg_lambda': 1.9826980964985924e-05}
  Trial   3 | Gini: 0.5392 | Params: {'max_depth': 6, 'learning_rate': 0.08810003129071789, 'n_estimators': 250, 'subsample': 0.7571172192068059, 'colsample

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8059264473611898
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegress

## 3.3 MLP

In [ ]:
# Optuna tuning — MLP (PyTorch)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FEATURES = X_train.shape[1]
MLP_EPOCHS = 15  # fixed training budget per fit (kept small for tuning)

# Fixed candidate architectures (hidden-layer sizes) keep the search space small.
MLP_ARCHITECTURES = {
    "128": [128],
    "256-128": [256, 128],
    "128-64": [128, 64],
}


class MLP(nn.Module):
    def __init__(self, n_features, hidden_sizes, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))  # single output logit
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def train_mlp(X_tr, y_tr, hidden_sizes, lr, weight_decay, batch_size, dropout):
    torch.manual_seed(SEED)
    model = MLP(N_FEATURES, hidden_sizes, dropout).to(device)
    # Fixed, shared class weight applied through the loss.
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(weight_ratio, device=device)
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay)

    ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    # shuffle=False keeps the time order of the (already ordered) training rows.
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    model.train()
    for _ in range(MLP_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def predict_mlp(model, X_va):
    model.eval()
    xb = torch.tensor(X_va, dtype=torch.float32).to(device)
    return torch.sigmoid(model(xb)).cpu().numpy()


# MLP tuning runs on a stratified, time-ordered subsample (see FRAC_MLP).
X_mlp, y_mlp = make_tuning_subsample(
    X_train, y_train, week_num_train, FRAC_MLP)
X_mlp_np = X_mlp.to_numpy(dtype=np.float32)
y_mlp_np = y_mlp.to_numpy(dtype=np.float32)


def mlp_objective(trial):
    arch_key = trial.suggest_categorical(
        "architecture", list(MLP_ARCHITECTURES))
    hidden_sizes = MLP_ARCHITECTURES[arch_key]
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    fold_ginis = []
    for tr_idx, va_idx in tscv.split(X_mlp_np):
        model = train_mlp(
            X_mlp_np[tr_idx], y_mlp_np[tr_idx],
            hidden_sizes, lr, weight_decay, batch_size, dropout,
        )
        proba = predict_mlp(model, X_mlp_np[va_idx])
        fold_ginis.append(gini(y_mlp_np[va_idx], proba))
    return float(np.mean(fold_ginis))


mlp_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
mlp_study.optimize(mlp_objective, n_trials=N_TRIALS, callbacks=[print_trial])

print("Best MLP params:", mlp_study.best_params)
print(f"Best MLP mean CV Gini: {mlp_study.best_value:.4f}")

# Retrain the final MLP on the FULL training block with the selected params.
best = mlp_study.best_params
mlp_best = train_mlp(
    X_train.to_numpy(dtype=np.float32),
    y_train.to_numpy(dtype=np.float32),
    MLP_ARCHITECTURES[best["architecture"]],
    best["learning_rate"],
    best["weight_decay"],
    best["batch_size"],
    best["dropout"],
)

## 3.4 Summary + save models

In [ ]:
# ── Tuning summary ──────────────────────────────────────────────────────────
print(f"Fixed weight_ratio (shared across all models): {weight_ratio:.4f}")
print(f"Random seed: {SEED}\n")
print("Best hyperparameters per model")
print("  LR :", lr_study.best_params, f"(CV Gini {lr_study.best_value:.4f})")
print("  XGB:", xgb_study.best_params, f"(CV Gini {xgb_study.best_value:.4f})")
print("  MLP:", mlp_study.best_params, f"(CV Gini {mlp_study.best_value:.4f})")

In [ ]:
import os
import joblib

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Logistic Regression — plain pickle via joblib
lr_path = os.path.join(MODELS_DIR, "lr_best.joblib")
joblib.dump(lr_best, lr_path)

# XGBoost — native format (portable across xgboost/sklearn versions)
xgb_path = os.path.join(MODELS_DIR, "xgb_best.json")
xgb_best.save_model(xgb_path)

# MLP — state dict + the architecture metadata needed to reconstruct the model
mlp_path = os.path.join(MODELS_DIR, "mlp_best.pt")
mlp_arch = MLP_ARCHITECTURES[mlp_study.best_params["architecture"]]
torch.save(
    {
        "state_dict": mlp_best.state_dict(),
        "n_features": N_FEATURES,
        "hidden_sizes": mlp_arch,
        "dropout": mlp_study.best_params["dropout"],
    },
    mlp_path,
)

print(f"Saved LR  -> {lr_path}")
print(f"Saved XGB -> {xgb_path}")
print(f"Saved MLP -> {mlp_path}")